# AfriVoices East Africa — ASR multilingue (édge) + KenLM

**Pipeline complet** : entraînement d'un modèle acoustique w2v-BERT 2.0 + CTC sur 6 langues
est-africaines (swa, kik, luo, kln, mas, som), puis décodage amélioré par un modèle de
langue KenLM par langue (rescoring n-gram via pyctcdecode).

**Organisation du notebook**
- §1–§7  : préparation (dépendances, config, données, vocabulaire, tokenizer, mélange, collator)
- §8–§11 : modèle, entraînement, évaluation WER, quantification INT8, rapport de latence
- §12    : inférence greedy → submission (baseline)
- §13    : KenLM par langue → submission améliorée (le gain principal)
- §14    : notes & dépôt conforme

**Contraintes compétition** : modèle ≤ 1 Md params, RAM ≤ 8 Go, CPU/edge, RTF ≤ 2×, 100 %
hors-ligne, code + poids publics sous licence permissive.

## 1. Dépendances

In [ ]:
# Sur Kaggle/Colab. Versions épinglées pour la reproductibilité (exigée par les règles).
!pip install -q "transformers>=4.44,<5" "datasets>=3.5,<4" "accelerate>=0.33" "jiwer>=3.0" "evaluate>=0.4" torchaudio soundfile ffmpeg-python librosa huggingface_hub

> ⚠️ **Après cette installation, redémarre le runtime** (Exécution → Redémarrer la session) avant de continuer : Colab garde sinon l'ancienne version de `datasets` en mémoire.

## 2. Configuration

Tout ce qui se règle est ici. `TEMPERATURE` est ton levier principal de rééquilibrage :
- T = 1.0  → distribution naturelle (Swahili domine ; bon si le test est Swahili-lourd / WER global)
- T = 1.3  → compromis recommandé au départ
- T = 2.0  → fort rééquilibrage (à réserver si le test est équilibré par langue)

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

import os, numpy as np, torch

BASE_MODEL    = "facebook/w2v-bert-2.0"   # MIT — conforme
SAMPLING_RATE = 16_000
TEMPERATURE   = 1.3
OUTPUT_DIR    = "/content/drive/MyDrive/afrivoices/w2vbert-ctc-v2"  # sur Drive (persistant)
MAX_AUDIO_S   = 20
SEED          = 42

# Les datasets font 200 Go - 1 To : on NE télécharge pas tout.
# Streaming + plafond d'échantillons par langue pour rester tenable sur Kaggle/Colab.
USE_STREAMING        = True
MAX_SAMPLES_PER_LANG = 6000      # ~150-250h selon durée moyenne ; monte si tu as le budget
ADD_SOMALI_MOGADISHU = False     # True pour ajouter le dialecte Mogadishu (repo Afrivoice)

# Heures officielles -> temperature sampling
LANG_HOURS = {"swa": 2979, "som": 1002, "kik": 754, "luo": 723, "kln": 521, "mas": 505}
LANGS = list(LANG_HOURS.keys())

torch.manual_seed(SEED); np.random.seed(SEED)

def temperature_probs(hours: dict, T: float) -> dict:
    h = np.array(list(hours.values()), dtype=float)
    w = h ** (1.0 / T); w = w / w.sum()
    return dict(zip(hours.keys(), w))

probs = temperature_probs(LANG_HOURS, TEMPERATURE)
print("Probabilités d'échantillonnage :", {k: round(v,3) for k,v in probs.items()})

## 3. Chargement des données (loaders réels, basés disque)

Deux structures validées, chacune écrite sur disque (seuls les **chemins** restent en RAM → pas de saturation mémoire) :
- **Swahili** `DigitalUmuganda/Afrivoice_Swahili` : `tar.xz` + manifestes `jsonl`, audio `.webm` → converti en wav 16k via ffmpeg.
- **anv_data_ke** `MCAA1-MSU/anv_data_ke` : parquet (audio WAV + `transcription` + `language`), 5 langues.

> Accepte les conditions *gated* sur les pages HF puis `login()`. Datasets en CC-BY-4.0 → attribution dans la model card.

In [ ]:
from huggingface_hub import login
login("hf_...")  # champ masqué — ne colle jamais ton token en clair

In [ ]:
import os, json, tarfile, subprocess
from huggingface_hub import hf_hub_download
from datasets import Dataset, Audio

SWA_REPO = "DigitalUmuganda/Afrivoice_Swahili"
SWA_WORK = "/content/swa"; os.makedirs(SWA_WORK, exist_ok=True)
SWA_TEXT = ["normalized_transcription", "transcription"]

def webm_to_wav(src, dst):
    subprocess.run(["ffmpeg","-y","-i",src,"-ar","16000","-ac","1",dst,"-loglevel","error"], check=True)

def load_swahili(domains=("agriculture","health","education","financial","government"),
                 split="train", shards_per_domain=1, max_per_shard=1200):
    rows = []
    for dom in domains:
        folder = f"{dom}_swahili_{split}"
        for n in range(shards_per_domain):
            try:
                man  = hf_hub_download(SWA_REPO, f"{folder}/manifest_{n}.jsonl", repo_type="dataset")
                tarp = hf_hub_download(SWA_REPO, f"{folder}/audio/audio_{n}.tar.xz", repo_type="dataset")
            except Exception as e:
                print("skip", folder, n, "->", str(e)[:80]); continue
            want = {}
            with open(man, encoding="utf-8") as f:
                for line in f:
                    if not line.strip(): continue
                    d = json.loads(line)
                    txt = next((d[k] for k in SWA_TEXT if d.get(k)), None)
                    key = d.get("key") or os.path.splitext(os.path.basename(d.get("audio_filepath","")))[0]
                    if txt and key: want[key] = txt.strip()
            ex = os.path.join(SWA_WORK, folder, f"s{n}"); os.makedirs(ex, exist_ok=True)
            done = 0
            with tarfile.open(tarp, "r:xz") as t:
                for m in t:
                    if done >= max_per_shard: break
                    if not m.isfile() or not m.name.endswith(".webm"): continue
                    key = os.path.splitext(os.path.basename(m.name))[0]
                    if key not in want: continue
                    fobj = t.extractfile(m)
                    if fobj is None: continue
                    webm = os.path.join(ex, key+".webm"); wav = os.path.join(ex, key+".wav")
                    with open(webm,"wb") as o: o.write(fobj.read())
                    try:
                        webm_to_wav(webm, wav); os.remove(webm)
                    except Exception:
                        continue
                    rows.append({"audio": wav, "text": want[key], "lang": "swa"}); done += 1
            print(f"{folder} s{n}: +{done} (total {len(rows)})")
    return Dataset.from_list(rows).cast_column("audio", Audio(sampling_rate=16000))

swa = load_swahili(shards_per_domain=1, max_per_shard=1200)
print(swa)

In [ ]:
import os, soundfile as sf
from datasets import load_dataset, Audio, Dataset

KE_REPO  = "MCAA1-MSU/anv_data_ke"
KE_LANGS = ["kik", "luo", "kln", "mas", "som"]
KE_OUT   = "/content/ke_wav"; os.makedirs(KE_OUT, exist_ok=True)

def load_ke(n_per_type=300, types=("unscripted","scripted")):
    paths, texts, langs = [], [], []
    for code in KE_LANGS:
        for typ in types:
            patt = f"hf://datasets/{KE_REPO}/{code}/train/{typ}/audios/*.parquet"
            try:
                ds = load_dataset("parquet", data_files=patt, split="train", streaming=True)
                ds = ds.cast_column("audio", Audio(sampling_rate=16000))
            except Exception as e:
                print("skip", code, typ, "->", str(e)[:80]); continue
            it = iter(ds); got = 0; bad = 0
            while got < (n_per_type[code] if isinstance(n_per_type, dict) else n_per_type):
                try:
                    ex = next(it)
                except StopIteration:
                    break
                except Exception:
                    bad += 1
                    if bad > 50: break
                    continue
                try:
                    txt = (ex.get("transcription") or "").strip()
                    if not txt: continue
                    a = ex["audio"]
                    fp = os.path.join(KE_OUT, f"{code}_{typ}_{got}.wav")
                    sf.write(fp, a["array"], a["sampling_rate"])
                    paths.append(fp); texts.append(txt); langs.append(code); got += 1
                except Exception:
                    bad += 1; continue
            print(f"{code}/{typ}: +{got} (sautes {bad}, total {len(paths)})")
    return Dataset.from_dict({"audio": paths, "text": texts, "lang": langs}).cast_column("audio", Audio(sampling_rate=16000))

ke = load_ke(n_per_type={"kln":2500, "mas":2500, "luo":2500, "kik":1500, "som":1500})
print(ke)

In [ ]:
# Un dataset par langue -> consomme par la suite du notebook (vocab, sampling, eval)
datasets_norm = {"swa": swa}
for code in ["kik","luo","kln","mas","som"]:
    datasets_norm[code] = ke.filter(lambda x, c=code: x["lang"] == c)
for l, d in datasets_norm.items():
    print(l, len(d))

## 4. Vocabulaire caractère partagé

Un seul vocabulaire pour les 6 langues (niveau caractère). On lit uniquement la colonne texte
(pas de décodage audio). `clean_text` sert aussi à la préparation et à l'inférence.

In [ ]:
import json, re
from collections import Counter

def clean_text(t: str) -> str:
    t = (t or "").lower().strip()
    t = re.sub(r"[^\w\sĩũáàâäéèêëíìîïóòôöúùûü']", "", t, flags=re.UNICODE)
    t = re.sub(r"\s+", " ", t)
    return t

counter = Counter()
for l, d in datasets_norm.items():
    for t in d["text"]:          # accès colonne -> pas de décodage audio
        counter.update(clean_text(t))

chars = sorted([c for c in counter if c != " "])
vocab = {c: i for i, c in enumerate(chars)}
vocab["|"]     = len(vocab)      # délimiteur de mot
vocab["[UNK]"] = len(vocab)
vocab["[PAD]"] = len(vocab)
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)
print("Taille vocab :", len(vocab))
print("Caractères :", "".join(chars))

## 5. Tokenizer + feature extractor + processor

In [ ]:
from transformers import (Wav2Vec2CTCTokenizer, SeamlessM4TFeatureExtractor,
                          Wav2Vec2BertProcessor)

tokenizer = Wav2Vec2CTCTokenizer("vocab.json", unk_token="[UNK]",
                                 pad_token="[PAD]", word_delimiter_token="|")
feature_extractor = SeamlessM4TFeatureExtractor.from_pretrained(BASE_MODEL)
processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def prepare(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=SAMPLING_RATE).input_features[0]
    batch["input_length"] = len(audio["array"]) / SAMPLING_RATE
    batch["labels"] = tokenizer(clean_text(batch["text"])).input_ids
    return batch

prepared = {}
for l, d in datasets_norm.items():
    d = d.filter(lambda ex: ex["text"] and len(ex["text"].strip()) > 0)
    d = d.map(prepare, remove_columns=d.column_names, num_proc=2)
    d = d.filter(lambda ln: ln < MAX_AUDIO_S, input_columns=["input_length"])
    prepared[l] = d

## 6. Mélange multilingue (temperature sampling) + split eval

`interleave_datasets` avec `probabilities` applique directement nos poids T.
On garde un **dev set par langue** pour reproduire la métrique et surveiller les petites langues.

In [ ]:
from datasets import interleave_datasets

eval_per_lang = {}
train_parts = []
for l in LANGS:
    if l not in prepared:
        continue
    split = prepared[l].train_test_split(test_size=0.03, seed=SEED)
    train_parts.append(split["train"])
    eval_per_lang[l] = split["test"]

order = [l for l in LANGS if l in prepared]
train_ds = interleave_datasets(
    train_parts,
    probabilities=[probs[l] for l in order],
    seed=SEED,
    stopping_strategy="all_exhausted",
)
print("Train (interleaved):", train_ds)
for l, d in eval_per_lang.items():
    print("eval", l, len(d))

## 7. Data collator CTC

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, padding=True,
                                                     return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=True,
                                                    return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

## 8. Modèle + entraînement

w2v-BERT 2.0 (~580 M params, < 1 Md ✅) avec tête CTC. `freeze_feature_encoder` au début
stabilise et accélère ; tu peux dégeler ensuite pour grappiller du WER.

In [ ]:
from transformers import Wav2Vec2BertForCTC, TrainingArguments, Trainer
import evaluate, numpy as np, torch
wer_metric = evaluate.load("wer")

model = Wav2Vec2BertForCTC.from_pretrained(
    BASE_MODEL,
    vocab_size=len(processor.tokenizer),
    pad_token_id=processor.tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,        # evite les nan quand label > audio sous-echantillonne
    add_adapter=True,
)
model.config.use_cache = False
n_params = sum(p.numel() for p in model.parameters())
print(f"Params: {n_params/1e6:.1f} M  ({'OK <1Md' if n_params < 1e9 else 'TROP GROS'})")

# argmax a la volee -> evite d'accumuler tous les logits en memoire pendant l'eval
def preprocess_logits_for_metrics(logits, labels):
    return logits.argmax(dim=-1)

def compute_metrics(pred):
    pred_ids = pred.predictions
    labels = pred.label_ids
    labels[labels == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(labels, group_tokens=False)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

from datasets import concatenate_datasets
eval_all = concatenate_datasets(list(eval_per_lang.values()))

# --- reglages auto selon le GPU (L4 ~22 Go vs A100 ~40 Go) ---
cuda   = torch.cuda.is_available()
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if cuda else 0
big    = gpu_gb > 30                                   # A100/A6000...
train_bs = 8 if big else 2
accum    = 2 if big else 8                             # batch effectif = 16 dans les deux cas
use_ckpt = not big                                     # checkpointing surtout utile si peu de VRAM
use_bf16 = cuda and torch.cuda.is_bf16_supported()     # bf16 sur A100, sinon fp16
print(f"GPU ~{gpu_gb:.0f} Go | batch={train_bs} accum={accum} checkpoint={use_ckpt} bf16={use_bf16}")

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=train_bs,
    gradient_accumulation_steps=accum,
    per_device_eval_batch_size=train_bs,
    eval_accumulation_steps=4,
    gradient_checkpointing=use_ckpt,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=5e-5,
    warmup_ratio=0.1,
    num_train_epochs=3,                                # 3 passes sur les donnees (s'adapte au volume)
    bf16=use_bf16, fp16=(cuda and not use_bf16),
    eval_strategy="steps", eval_steps=500,
    save_steps=500, save_total_limit=2,
    logging_steps=50,
    load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
    report_to="none", seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_all,
                  data_collator=data_collator, compute_metrics=compute_metrics,
                  preprocess_logits_for_metrics=preprocess_logits_for_metrics,
                  processing_class=processor)

import os
from transformers.trainer_utils import get_last_checkpoint
last_ckpt = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
trainer.train(resume_from_checkpoint=last_ckpt)   # None = depart de zero ; sinon reprend
trainer.save_model(OUTPUT_DIR); processor.save_pretrained(OUTPUT_DIR)

## 9. WER par langue (ta boussole)

Le leaderboard donne un WER global, mais surveiller le **par-langue** t'évite de découvrir
trop tard qu'une petite langue est cassée. On calcule aussi le **macro-WER** au cas où.

In [ ]:
@torch.no_grad()
def transcribe_batch(model, feats):
    model.eval()
    inp = processor.feature_extractor.pad(
        [{"input_features": f} for f in feats], padding=True, return_tensors="pt")
    logits = model(**{k: v.to(model.device) for k, v in inp.items()}).logits
    ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(ids)

results = {}
for l, d in eval_per_lang.items():
    preds, refs = [], []
    for i in range(0, len(d), 16):
        chunk = d[i:i+16]
        preds += transcribe_batch(model, chunk["input_features"])
        refs  += processor.batch_decode(
            [[t if t != -100 else processor.tokenizer.pad_token_id for t in lab]
             for lab in chunk["labels"]], group_tokens=False)
    results[l] = wer_metric.compute(predictions=preds, references=refs)

print("WER par langue :")
for l, w in results.items():
    print(f"  {l}: {w:.3f}")
print(f"Macro-WER : {np.mean(list(results.values())):.3f}")

## 10. Quantification INT8 (CPU / edge)

Quantification dynamique des couches `Linear` → ~2× moins de mémoire, inférence CPU rapide,
100 % hors-ligne. C'est le format que tu mesureras pour le rapport hardware.
(Piste avancée : export ONNX via `optimum-cli export onnx` + ORT INT8 si tu veux plus de débit.)

In [ ]:
import gc, torch, os
from transformers import Wav2Vec2BertForCTC

# 1) Libere le GPU (entrainement) : la quantification INT8 est une operation CPU
for v in ["trainer", "model", "cpu_model", "model_int8"]:
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()

# 2) Recharge le modele entraine sur CPU (depuis OUTPUT_DIR, ou un dossier Drive)
MODEL_DIR = OUTPUT_DIR
cpu_model = Wav2Vec2BertForCTC.from_pretrained(MODEL_DIR).to("cpu").eval()

# 3) Quantification dynamique INT8 (couches Linear)
model_int8 = torch.quantization.quantize_dynamic(
    cpu_model, {torch.nn.Linear}, dtype=torch.qint8)
torch.save(model_int8.state_dict(), os.path.join(MODEL_DIR, "model_int8.pt"))

import io
buf = io.BytesIO(); torch.save(model_int8.state_dict(), buf)
print(f"Taille poids INT8 : {buf.tell()/1e6:.1f} MB")

## 11. Rapport de validation hardware (latence) — LIVRABLE OBLIGATOIRE

La règle 10 exige la latence d'inférence sur **tout le test set**. On mesure le temps mur
par énoncé sur **CPU** et on calcule le RTF (temps / durée audio ; cible ≤ 2×).
Exécute idéalement cette cellule sur un Raspberry Pi 4 / smartphone pour le rapport final.

In [ ]:
import time, csv, torch

torch.set_num_threads(os.cpu_count())   # CPU-only

# Remplace 'eval_all' par le vrai test set du concours une fois chargé/préparé.
test_ds = eval_all
rows, total_audio, total_proc = [], 0.0, 0.0
for i, ex in enumerate(test_ds):
    feats = [ex["input_features"]]
    audio_s = ex.get("input_length", len(ex["input_features"]) * 0.02)
    t0 = time.perf_counter()
    _ = transcribe_batch(model_int8.to("cpu"), feats)
    dt = time.perf_counter() - t0
    total_audio += audio_s; total_proc += dt
    rows.append({"idx": i, "audio_s": round(audio_s, 3),
                 "proc_s": round(dt, 3), "rtf": round(dt / max(audio_s, 1e-6), 3)})

with open(os.path.join(OUTPUT_DIR, "latency_report.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["idx", "audio_s", "proc_s", "rtf"]); w.writeheader()
    w.writerows(rows)

print(f"RTF moyen : {total_proc/total_audio:.3f}  ({'OK <=2x' if total_proc/total_audio <= 2 else 'TROP LENT'})")
print(f"Params: {sum(p.numel() for p in cpu_model.parameters())/1e6:.0f}M  |  RAM cible <= 8GB")
print("-> latency_report.csv généré")

## 12. Inférence test → submission.csv (baseline greedy)

Pipeline 100 % automatique. On charge le modèle entraîné, on transcrit les parquets de
test par **argmax glouton** (sans modèle de langue), et on écrit le fichier au format Kaggle.

**Format Kaggle** : 3 colonnes `id,language,transcription`, code langue ISO 639-3.

**Décodage audio robuste** : les parquets de test stockent le WAV soit en bytes bruts,
soit **encodé en base64** (cas du somali de test). `decode_robuste` gère les deux + repli
ffmpeg. Sans ça, tout le somali de test serait perdu.

> Note : ce §12 est la *baseline*. Le §13 (KenLM) produit un meilleur score et écrit le
> même `submission.csv`. Pour la soumission finale, utilise le §13.

### 12.1 — Montage Drive + vérification des chemins

In [ ]:
# === Montage Drive + vérification des chemins ===
from google.colab import drive
import os, glob
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/afrivoices/w2vbert-ctc-v2"   # modèle entraîné
TEST_ROOT  = "/content/drive/MyDrive/afrivoices/test"            # parquets de test

print("Modèle existe :", os.path.isdir(OUTPUT_DIR))
files = sorted(glob.glob(os.path.join(TEST_ROOT, "**", "*.parquet"), recursive=True))
print("Parquets test trouvés :", len(files))

### 12.2 — Fonction de décodage audio robuste (réutilisée partout)

Décode les bytes audio quel que soit leur format. Cette fonction est utilisée par
l'inférence greedy (§12) ET par l'inférence KenLM (§13).

In [ ]:
import io, base64, os, numpy as np, soundfile as sf, librosa

def decode_robuste(a):
    """WAV bytes bruts OU base64 OU autre format via ffmpeg -> array 16k mono float32."""
    b = a.get("bytes")
    if not b:
        raise ValueError("bytes vides")
    try:                                   # 1) bytes WAV bruts
        arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception:
        try:                               # 2) base64 -> WAV
            arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
        except Exception:                  # 3) ffmpeg (mp3/ogg/webm/wav cassé)
            import tempfile
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as t:
                try: t.write(base64.b64decode(b))
                except Exception: t.write(b)
                p = t.name
            arr, sr = librosa.load(p, sr=16000, mono=True); os.unlink(p)
            return arr.astype(np.float32)
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

print("decode_robuste prêt")

### 12.3 — Inférence greedy + écriture submission.csv

Batch dynamique (trie par durée pour minimiser le padding, ajuste la taille du batch
selon la longueur → évite les OOM GPU). Décodage par `argmax`. Écrit aussi
`latency_report.csv` (livrable obligatoire : id, langue, durée, temps, RTF).

In [ ]:
import glob, time, csv, gc, torch
from datasets import load_dataset
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from collections import Counter

OUTPUT_DIR = "/content/drive/MyDrive/afrivoices/w2vbert-ctc-v2"
TEST_ROOT  = "/content/drive/MyDrive/afrivoices/test"
SUB_OUT    = "/content/drive/MyDrive/afrivoices/submission.csv"
LAT_OUT    = "/content/drive/MyDrive/afrivoices/latency_report.csv"
OUT_COL    = "transcription"
MAX_TOK    = 8 * 20.0

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Wav2Vec2BertForCTC.from_pretrained(OUTPUT_DIR).to(device).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(OUTPUT_DIR)

torch.cuda.empty_cache()
files = sorted(glob.glob(os.path.join(TEST_ROOT, "**", "*.parquet"), recursive=True))
test = load_dataset("parquet", data_files=files, split="train")
has_lang = "language" in test.column_names
print(len(test), "exemples")

print("Décodage durées + tri...")
items = []
for idx in range(len(test)):
    try: items.append((idx, len(decode_robuste(test[idx]["audio"]))/16000))
    except Exception: items.append((idx, -1.0))
items.sort(key=lambda x: x[1])

sub_rows, lat_rows = [], []
total_audio = total_proc = 0.0
n_skip = done = 0
i = 0
while i < len(items):
    longest = items[i][1]
    if longest <= 0:
        sub_rows.append({"id": test[items[i][0]]["id"], OUT_COL: ""}); n_skip+=1; i+=1; done+=1; continue
    bs = min(16, max(1, int(MAX_TOK // max(longest,1.0))))
    batch_idx = [items[k][0] for k in range(i, min(i+bs,len(items))) if items[k][1] > 0]
    i += bs
    arrays, ids, langs, durs = [], [], [], []
    for idx in batch_idx:
        row = test[idx]
        try: arr = decode_robuste(row["audio"])
        except Exception:
            sub_rows.append({"id": row["id"], OUT_COL: ""}); n_skip+=1; continue
        arrays.append(arr); ids.append(row["id"])
        langs.append(row["language"] if has_lang else ""); durs.append(len(arr)/16000)
    if not arrays: continue
    feats = processor.feature_extractor(arrays, sampling_rate=16000, return_tensors="pt", padding=True)
    feats = {k: v.to(device) for k,v in feats.items()}
    t0 = time.perf_counter()
    try:
        with torch.no_grad(): logits = model(**feats).logits
        texts = processor.batch_decode(torch.argmax(logits, dim=-1))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); texts=[]
        for arr in arrays:
            f1={k:v.to(device) for k,v in processor.feature_extractor([arr],sampling_rate=16000,return_tensors="pt",padding=True).items()}
            with torch.no_grad(): lg=model(**f1).logits
            texts.append(processor.batch_decode(torch.argmax(lg,dim=-1))[0]); del f1,lg
    dt = time.perf_counter() - t0
    ba = sum(durs) or 1e-6
    for _id, txt, dur, lang in zip(ids, texts, durs, langs):
        sub_rows.append({"id": _id, OUT_COL: txt.strip()})
        sh = dt*(dur/ba)
        lat_rows.append({"id": _id, "language": lang, "audio_s": round(dur,3),
                         "proc_s": round(sh,4), "rtf": round(sh/max(dur,1e-6),3)})
    total_audio += sum(durs); total_proc += dt
    del feats
    done += len(batch_idx)
    if done % 4000 < bs:
        torch.cuda.empty_cache(); gc.collect(); print(f"  {done}/{len(test)}")

with open(SUB_OUT,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=["id",OUT_COL]); w.writeheader(); w.writerows(sub_rows)
with open(LAT_OUT,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=["id","language","audio_s","proc_s","rtf"]); w.writeheader(); w.writerows(lat_rows)
print(f"\n{len(sub_rows)} lignes ({n_skip} vides). RTF {total_proc/max(total_audio,1e-6):.3f}")
print("langues:", dict(Counter(r['language'] for r in lat_rows)))

### 12.4 — Assemblage du fichier final (3 colonnes + garde-fous)

Fusionne la langue, garantit aucune cellule vide (placeholder `_`), aucun NaN, colonnes
exactes. Un échec d'`assert` = fichier qui serait refusé par Kaggle.

In [ ]:
import pandas as pd, glob

sub = pd.read_csv("/content/drive/MyDrive/afrivoices/submission.csv")       # id + transcription
lat = pd.read_csv("/content/drive/MyDrive/afrivoices/latency_report.csv")   # id + language

m = sub.merge(lat[["id","language"]], on="id", how="left")
# langue manquante (audios vides absents du latency) -> récupérer depuis les parquets
if m["language"].isna().any():
    files = sorted(glob.glob("/content/drive/MyDrive/afrivoices/test/**/*.parquet", recursive=True))
    meta = pd.concat([pd.read_parquet(f, columns=["id","language"]) for f in files], ignore_index=True)
    m["language"] = m["language"].fillna(m["id"].map(dict(zip(meta["id"], meta["language"]))))

final = pd.DataFrame({"id": m["id"], "language": m["language"],
                      "transcription": m["transcription"].fillna("").astype(str)})

# nettoyage : retours à la ligne (cassent le CSV), placeholder pour vides
final["transcription"] = final["transcription"].str.replace(r"[\r\n]+", " ", regex=True).str.strip()
final.loc[final["transcription"]=="", "transcription"] = "_"

# garde-fous (un échec ici = fichier refusé par Kaggle)
assert final["id"].nunique() == len(final),               "ids dupliqués"
assert final.columns.tolist() == ["id","language","transcription"], "colonnes != attendu"
assert final["language"].isna().sum() == 0,               "langue manquante"
assert (final["transcription"].str.strip()=="").sum() == 0, "transcription vide"

OUT = "/content/drive/MyDrive/afrivoices/submission_final.csv"
final.to_csv(OUT, index=False)
print(f"OK -> {OUT}  ({len(final)} lignes)")
print("codes langue :", sorted(final["language"].unique()))
print(final["language"].value_counts())

## 13. KenLM par langue (rescoring n-gram) — le gain principal

Le modèle acoustique transforme l'audio en probabilités de caractères (logits). Le
décodage greedy prend bêtement le caractère le plus probable à chaque pas. Un **modèle de
langue n-gram** (KenLM) ajoute la connaissance « quels enchaînements de lettres/mots sont
plausibles dans cette langue », et corrige les fautes d'orthographe du modèle acoustique.

On entraîne **un KenLM par langue** sur les transcriptions d'entraînement (texte seul,
rapide), puis on décode chaque audio avec le LM de sa langue (colonne `language` du test).

**Important** : le KenLM ne modifie pas le modèle acoustique. C'est une couche de
post-traitement. Le modèle (`w2vbert-ctc-v2`) + les 6 fichiers `.bin` + ce code de
décodage forment ensemble le système complet, à conserver pour le dépôt.

*Gain mesuré sur le leaderboard : 0.566 → 0.469 (≈ −10 points de WER), uniquement grâce
au KenLM (modèle identique).*

### 13.1 — Installer KenLM (lecteur Python) + pyctcdecode

In [ ]:
!pip install -q https://github.com/kpu/kenlm/archive/master.zip pyctcdecode
print("ok")

### 13.2 — Extraire les transcriptions d'entraînement, par langue

5 langues kényanes depuis les parquets `anv_data_ke` (colonne `transcription`), texte
seul (aucun audio téléchargé → rapide). Connexion HF requise.

In [ ]:
import os, glob
from datasets import load_dataset
import pandas as pd
from huggingface_hub import login
# login("hf_...")   # décommente si pas déjà connecté dans cette session

os.makedirs("/content/drive/MyDrive/afrivoices/lm", exist_ok=True)

# --- 5 langues kényanes depuis anv_data_ke (parquet, colonne transcription) ---
KE = "MCAA1-MSU/anv_data_ke"
from huggingface_hub import HfFileSystem
fs = HfFileSystem()
texts = {l: [] for l in ["kik","luo","kln","mas","som"]}

for lang in texts:
    # tous les parquets train de cette langue
    pattern = f"datasets/{KE}/{lang}/train/**/*.parquet"
    paths = fs.glob(pattern)
    print(lang, ":", len(paths), "parquets")
    for p in paths:
        try:
            df = pd.read_parquet(f"hf://{p}", columns=["transcription"])
            texts[lang] += [t for t in df["transcription"].tolist()
                            if isinstance(t, str) and t.strip()]
        except Exception as e:
            print("  skip", p, e)
    print(f"  -> {len(texts[lang])} phrases")

### 13.3 — Extraire le swahili depuis les manifests

In [ ]:
# --- swahili depuis les manifests ---
from huggingface_hub import HfFileSystem
import json
fs = HfFileSystem()
SW = "DigitalUmuganda/Afrivoice_Swahili"
swa = []
manifests = fs.glob(f"datasets/{SW}/**/manifest_*.jsonl")
print("manifests swa:", len(manifests))
for mp in manifests:
    try:
        with fs.open(mp) as f:
            for line in f:
                d = json.loads(line)
                t = d.get("normalized_transcription") or d.get("transcription")
                if isinstance(t, str) and t.strip():
                    swa.append(t.strip())
    except Exception as e:
        print("  skip", mp, e)
texts["swa"] = swa
print("swa:", len(swa), "phrases")

### 13.4 — Normaliser le texte **exactement** comme à l'entraînement

`clean_text` (minuscules, ponctuation retirée) identique au §4. Crucial : si le texte du
LM n'est pas normalisé comme le modèle acoustique, le LM proposera des tokens que le
modèle ne peut pas produire.

In [ ]:
# --- normalisation identique à l'entraînement (clean_text de la §4) ---
import re
def clean_text(t):
    t = t.lower().strip()
    return re.sub(r"[^\w\s']", "", t, flags=re.UNICODE)

for lang in list(texts.keys()):
    texts[lang] = [c for t in texts[lang] if (c := clean_text(t))]
    print(f"{lang}: {len(texts[lang])} phrases après nettoyage")

### 13.5 — Écrire les 6 corpus texte sur le Drive

In [ ]:
# --- écriture des 6 corpus ---
import os
os.makedirs("/content/drive/MyDrive/afrivoices/lm", exist_ok=True)
for lang, txts in texts.items():
    path = f"/content/drive/MyDrive/afrivoices/lm/{lang}.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(txts))
    print(f"{lang}: {len(txts)} -> {path}")

### 13.6 — Compiler les binaires KenLM (lmplz, build_binary)

Le pip `kenlm` ne fournit que le lecteur Python ; le compilateur `lmplz` se construit
depuis les sources. À faire une seule fois par session.

In [ ]:
# compiler les binaires KenLM (lmplz, build_binary) une fois
!apt-get install -y build-essential cmake libboost-all-dev libeigen3-dev >/dev/null 2>&1
!git clone https://github.com/kpu/kenlm.git /content/kenlm >/dev/null 2>&1
!cd /content/kenlm && mkdir -p build && cd build && cmake .. >/dev/null 2>&1 && make -j4 >/dev/null 2>&1
!ls /content/kenlm/build/bin/

### 13.7 — Entraîner un 5-gram par langue → `.arpa` puis `.bin`

`--discount_fallback` rend l'estimation robuste sur les petits corpus (ex. mas, 54k
phrases). Le `.bin` charge bien plus vite que le `.arpa` au décodage.

In [ ]:
# entraîner un 5-gram par langue -> .arpa puis convertir en .bin (plus rapide à charger)
import os
LM = "/content/drive/MyDrive/afrivoices/lm"
BIN = "/content/kenlm/build/bin"
for lang in ["swa","kik","luo","kln","mas","som"]:
    txt, arpa, binf = f"{LM}/{lang}.txt", f"{LM}/{lang}.arpa", f"{LM}/{lang}.bin"
    os.system(f"{BIN}/lmplz -o 5 --discount_fallback <{txt}> {arpa} 2>/tmp/{lang}.log")
    os.system(f"{BIN}/build_binary {arpa} {binf} 2>>/tmp/{lang}.log")
    okb = os.path.exists(binf) and os.path.getsize(binf) > 0
    print(f"{lang}: {'OK' if okb else 'ÉCHEC'}  bin={os.path.getsize(binf) if okb else 0} o")
    if not okb:
        print("   --- log ---"); os.system(f"tail -5 /tmp/{lang}.log")

### 13.8 — Charger le modèle + construire le mapping des labels CTC

pyctcdecode exige **exactement un blank** (`""`) et **un séparateur de mot** (`" "`).
Pièges de ce tokenizer : le blank réel est `[PAD]`, le séparateur de mot est l'espace
`" "` (index 62, pas `|`), et les tokens spéciaux multi-caractères (`[UNK]`, `<s>`,
`</s>`) doivent devenir des placeholders **uniques** non vides (sinon doublons → erreur).
On vérifie `blank count == 1` et `space count == 1`.

In [ ]:
import torch
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder

OUTPUT_DIR = "/content/drive/MyDrive/afrivoices/w2vbert-ctc-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Wav2Vec2BertForCTC.from_pretrained(OUTPUT_DIR).to(device).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(OUTPUT_DIR)

vocab = processor.tokenizer.get_vocab()
raw = [t for t,_ in sorted(vocab.items(), key=lambda kv: kv[1])]
blank_tok = processor.tokenizer.pad_token   # '[PAD]'

labels = []
for t in raw:
    if t == blank_tok:        labels.append("")     # blank unique
    elif t in ("|", " "):     labels.append(" ")    # séparateur de mot
    elif t in {"[UNK]", "<s>", "</s>"}: labels.append("⁇")  # placeholder
    else:                     labels.append(t)

# rendre les placeholders uniques (sinon doublons interdits)
n = 0
for i,x in enumerate(labels):
    if x == "⁇":
        n += 1; labels[i] = "⁇"*n

assert labels.count("") == 1, "il faut exactement un blank"
assert labels.count(" ") == 1, "il faut exactement un espace"
assert len(labels) == len(set(labels)), "doublons interdits"
print("mapping OK | blank=1 space=1 | nb labels:", len(labels))

### 13.9 — Construire un décodeur KenLM par langue

`alpha` = poids du LM, `beta` = bonus de longueur. 0.5 / 1.0 = valeurs prudentes et
sûres. Les `unigrams` (mots du corpus) bornent la recherche → plus rapide et précis.

In [ ]:
LM = "/content/drive/MyDrive/afrivoices/lm"
def unigrams(lang, top=50000):
    from collections import Counter
    c = Counter()
    with open(f"{LM}/{lang}.txt", encoding="utf-8") as f:
        for line in f:
            c.update(line.split())
    return [w for w,_ in c.most_common(top)]

decoders = {}
for lang in ["swa","kik","luo","kln","mas","som"]:
    decoders[lang] = build_ctcdecoder(
        labels, kenlm_model_path=f"{LM}/{lang}.bin",
        unigrams=unigrams(lang), alpha=0.5, beta=1.0)
print("décodeurs prêts:", list(decoders.keys()))

### 13.10 — Test rapide (greedy vs +LM) sur 5 exemples

Vérifier que le +LM produit du texte plausible **avant** de lancer les 41 733. On veut
des mots mieux orthographiés et bien espacés, pas du charabia.

In [ ]:
import torch, glob
from datasets import load_dataset
files = sorted(glob.glob("/content/drive/MyDrive/afrivoices/test/**/*.parquet", recursive=True))
test = load_dataset("parquet", data_files=files[:1], split="train")

for k in range(5):
    row = test[k]
    arr = decode_robuste(row["audio"])
    feats = processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt", padding=True)
    feats = {kk: v.to(device) for kk, v in feats.items()}
    with torch.no_grad():
        logits = model(**feats).logits[0].cpu().numpy()
    greedy = processor.batch_decode(torch.argmax(torch.tensor(logits)[None], dim=-1))[0]
    lang = row["language"]
    lm = decoders[lang].decode(logits)
    print(f"\n[{lang}] greedy: {greedy!r}")
    print(f"[{lang}] +LM   : {lm!r}")

### 13.11 — Inférence complète avec décodage KenLM → submission.csv

Même boucle qu'au §12 mais on remplace l'argmax par `decoder.decode(logits)` selon la
langue. Le décodage LM tourne sur CPU (plus lent que l'argmax) mais le RTF reste très
sous 2×.

In [ ]:
import glob, os, time, csv, io, gc, base64, torch, numpy as np
import soundfile as sf, librosa
from datasets import load_dataset
from collections import Counter

TEST_ROOT = "/content/drive/MyDrive/afrivoices/test"
SUB_OUT   = "/content/drive/MyDrive/afrivoices/submission.csv"
LAT_OUT   = "/content/drive/MyDrive/afrivoices/latency_report.csv"
OUT_COL   = "transcription"
MAX_TOK   = 8 * 20.0

# decode_robuste doit être défini (sinon recharge la cellule précédente)
torch.cuda.empty_cache()
files = sorted(glob.glob(os.path.join(TEST_ROOT, "**", "*.parquet"), recursive=True))
test = load_dataset("parquet", data_files=files, split="train")
has_lang = "language" in test.column_names
print(len(test), "exemples")

print("Décodage durées + tri...")
items = []
for idx in range(len(test)):
    try: items.append((idx, len(decode_robuste(test[idx]["audio"]))/16000))
    except Exception: items.append((idx, -1.0))
items.sort(key=lambda x: x[1])

sub_rows, lat_rows = [], []
total_audio = total_proc = 0.0
n_skip = done = 0
i = 0
while i < len(items):
    longest = items[i][1]
    if longest <= 0:
        sub_rows.append({"id": test[items[i][0]]["id"], OUT_COL: ""}); n_skip+=1; i+=1; done+=1; continue
    bs = min(16, max(1, int(MAX_TOK // max(longest,1.0))))
    batch_idx = [items[k][0] for k in range(i, min(i+bs,len(items))) if items[k][1] > 0]
    i += bs
    arrays, ids, langs, durs = [], [], [], []
    for idx in batch_idx:
        row = test[idx]
        try: arr = decode_robuste(row["audio"])
        except Exception:
            sub_rows.append({"id": row["id"], OUT_COL: ""}); n_skip+=1; continue
        arrays.append(arr); ids.append(row["id"])
        langs.append(row["language"] if has_lang else "swa"); durs.append(len(arr)/16000)
    if not arrays: continue
    feats = processor.feature_extractor(arrays, sampling_rate=16000, return_tensors="pt", padding=True)
    feats = {k: v.to(device) for k,v in feats.items()}
    t0 = time.perf_counter()
    with torch.no_grad():
        logits = model(**feats).logits.cpu().numpy()    # (B, T, V)
    # décodage LM par langue, individuel
    texts = []
    for bi in range(len(arrays)):
        dec = decoders.get(langs[bi], decoders["swa"])
        texts.append(dec.decode(logits[bi]).strip())
    dt = time.perf_counter() - t0
    ba = sum(durs) or 1e-6
    for _id, txt, dur, lang in zip(ids, texts, durs, langs):
        sub_rows.append({"id": _id, OUT_COL: txt})
        sh = dt*(dur/ba)
        lat_rows.append({"id": _id, "language": lang, "audio_s": round(dur,3),
                         "proc_s": round(sh,4), "rtf": round(sh/max(dur,1e-6),3)})
    total_audio += sum(durs); total_proc += dt
    del feats
    done += len(batch_idx)
    if done % 2000 < bs:
        torch.cuda.empty_cache(); gc.collect()
        print(f"  {done}/{len(test)}")

with open(SUB_OUT,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=["id",OUT_COL]); w.writeheader(); w.writerows(sub_rows)
with open(LAT_OUT,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=["id","language","audio_s","proc_s","rtf"]); w.writeheader(); w.writerows(lat_rows)
print(f"\n{len(sub_rows)} lignes ({n_skip} vides). RTF {total_proc/max(total_audio,1e-6):.3f}")
print("langues:", dict(Counter(r['language'] for r in lat_rows)))

### 13.12 — Assemblage final (identique au §12.4) → submission_final.csv

In [ ]:
import pandas as pd, glob

sub = pd.read_csv("/content/drive/MyDrive/afrivoices/submission.csv")
lat = pd.read_csv("/content/drive/MyDrive/afrivoices/latency_report.csv")

m = sub.merge(lat[["id","language"]], on="id", how="left")
if m["language"].isna().any():
    files = sorted(glob.glob("/content/drive/MyDrive/afrivoices/test/**/*.parquet", recursive=True))
    meta = pd.concat([pd.read_parquet(f, columns=["id","language"]) for f in files], ignore_index=True)
    m["language"] = m["language"].fillna(m["id"].map(dict(zip(meta["id"], meta["language"]))))

final = pd.DataFrame({"id": m["id"], "language": m["language"],
                      "transcription": m["transcription"].fillna("").astype(str)})
final["transcription"] = final["transcription"].str.replace(r"[\r\n]+", " ", regex=True).str.strip()
final.loc[final["transcription"]=="", "transcription"] = "_"

assert final["id"].nunique() == len(final)
assert final.columns.tolist() == ["id","language","transcription"]
assert final["language"].isna().sum() == 0
assert (final["transcription"].str.strip()=="").sum() == 0

OUT = "/content/drive/MyDrive/afrivoices/submission_final.csv"
final.to_csv(OUT, index=False)
print(f"OK -> {OUT}  ({len(final)} lignes)")
print(final["language"].value_counts())

## 14. Notes, dépôt conforme & pistes

**Système complet à conserver / déposer** (les 3 ensemble) :
1. les poids du modèle acoustique (`w2vbert-ctc-v2/`),
2. les 6 KenLM (`lm/*.bin`) — ou les corpus + le script §13 qui les régénère,
3. le code de décodage (mapping labels §13.8 + décodeurs §13.9 + inférence §13.11).

Sans les KenLM, recharger le modèle seul donne le résultat **greedy** (WER plus élevé).

**Structure du dépôt public** (licence permissive exigée) :
```
afrivoices-asr/
├── README.md          # how-to, scores, choix de design
├── MODEL_CARD.md      # langues, données, limites, biais
├── LICENSE            # MIT / Apache-2.0
├── requirements.txt   # versions épinglées
├── train.ipynb        # ce notebook
├── lm/                # KenLM .bin (ou script de régénération)
├── latency_report.csv # rapport hardware
└── checkpoints/       # poids (ou lien HF)
```

**Pistes pour baisser encore le macro-WER :**
- **Régler alpha/beta du KenLM** sur le split eval interne (gain rapide, ne pas régler à
  l'aveugle sur le leaderboard).
- **Plus de données som / kln / mas à l'entraînement.** ⚠️ Correction : contrairement à
  une hypothèse initiale, le base64 ne concernait **que le test**, pas l'entraînement
  (mesuré : 149/150 bytes bruts OK côté train). Le somali n'était donc pas « perdu » —
  il était simplement **sous-utilisé** (~1500 clips sur 81 490 disponibles). Le levier
  est d'**augmenter le volume** de ces langues faibles, pas de corriger un bug de loader.
- **Réentraînement** : à faire dans un notebook séparé (chargement depuis le Drive) pour
  économiser les unités de calcul. Repartir du checkpoint v2 (fine-tuning) plutôt que de
  zéro, et écrire dans un dossier distinct (`w2vbert-ctc-v3`) pour préserver v2.